In [9]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from itertools import combinations
from rapidfuzz import fuzz

In [10]:
seed = 1
filepath_syn = Path(f"../data/synthetic/{seed}")
filepath_gt = Path(f"../data/ground_truth/{seed}")

customers_raw = pd.read_csv(filepath_syn/"customers_raw.csv")
products_raw = pd.read_csv(filepath_syn/"products_raw.csv")
orders_raw = pd.read_csv(filepath_syn/"orders_raw.csv")

customers_clean = pd.read_csv(filepath_gt/"customers_clean.csv")
products_clean = pd.read_csv(filepath_gt/"products_clean.csv")

In [11]:
def show_all(df, name=""):
    with pd.option_context("display.max_rows", None,
                           "display.max_columns", None,
                           "display.width", None):
        if name:
            print(f"=== {name}: {len(df)} rows ===")
        display(df)


def inspect_duplicates(df, subset=None, name=""):
    dups = df[df.duplicated(subset=subset, keep=False)]
    sort_cols = subset if subset else list(df.columns)
    with pd.option_context("display.max_rows", None,
                           "display.max_columns", None):
        print(f"=== {name}: {len(dups)} duplicate rows "
              f"(subset={subset or 'all columns'}) ===")
        display(dups.sort_values(by=sort_cols))

def inspect_fuzzy_duplicates(df, cols=None, exclude=None, threshold=85,
                             scorer=fuzz.ratio, name=""):
    exclude = set(exclude or [])
    if cols is None:                      
        cols = [c for c in df.columns if c not in exclude]
    sub = df[cols].fillna("").astype(str)
    recs = sub.to_dict("records")
    idx = df.index.tolist()

    rows = []
    for a, b in combinations(range(len(df)), 2):
        scores = {c: scorer(recs[a][c], recs[b][c]) for c in cols}
        avg = sum(scores.values()) / len(cols)
        if avg >= threshold:
            rows.append({"score": round(avg, 1), "idx_a": idx[a], "idx_b": idx[b],
                         **{f"{c}_a": recs[a][c] for c in cols},
                         **{f"{c}_b": recs[b][c] for c in cols}})

    result = (pd.DataFrame(rows).sort_values("score", ascending=False)
              .reset_index(drop=True))
    with pd.option_context("display.max_rows", None, "display.max_columns", None):
        print(f"=== {name}: {len(result)} fuzzy pairs "
              f"(threshold={threshold}, cols={cols}) ===")
        display(result)
    return result

In [21]:
# Customers

#show_all(customers_raw.sort_values("customer_id"), "customers_raw")             #sorted raw
#show_all(customers_clean.sort_values("customer_id"), "customers_clean")         #sorted clean
#show_all(customers_raw, "customers_raw")                                        #unsorted
show_all(customers_clean, "customers_clean")                                    #unsorted

# Duplicates

#inspect_duplicates(customers_raw, name="customers_raw")
#inspect_duplicates(customers_clean, name="customers_clean")    # 0 expected

#inspect_fuzzy_duplicates(customers_raw, cols=["full_name", "email"], threshold=80, name="customers_raw");

=== customers_clean: 500 rows ===


,customer_id,full_name,email,country_code,registered_at
0,1,Karsten Wieloch-Bolnbach,erkan77@example.com,CH,2025-07-17
1,2,Rebecca Häring-Dehmel,ybaum@example.com,GB,2025-03-02
2,3,Nico Matthäi,irmtraud39@example.org,AT,2022-09-13
3,4,Aldo Bauer,kranzheidelinde@example.org,IT,2023-08-04
4,5,Jose Hartung,joderwald@example.org,AT,2022-12-10
5,6,Univ.Prof. Emanuel Wesack MBA.,susan06@example.net,BE,2025-06-18
6,7,Dogan Schuchhardt,erol58@example.com,BE,2024-04-26
7,8,Gereon Hölzenbecher MBA.,cschacht@example.com,BE,2022-01-24
8,9,Herlinde Stadelmann-Gerlach,bertha51@example.com,NL,2025-08-24
9,10,Reinhard Fröhlich,fkoch-ii@example.net,FR,2020-04-01


In [13]:
# Products

show_all(products_raw.sort_values("product_id"), "products_raw")     #sorted
show_all(products_clean.sort_values("product_id"), "products_clean") 
#show_all(products, "products")                               #unsorted

# Duplicates

inspect_duplicates(products_raw, name="products_raw")
#inspect_duplicates(products_clean, subset=["product_id"], name="products_clean")

=== products_raw: 100 rows ===


,product_id,name,category,price_eur,in_stock
0,1,Object-based explicit attitude,Clothing,134.94 EUR,yes
1,2,Configurable upward-trending complexity,Clothing,€498.06,1
2,3,Triple-buffered global hierarchy,Books,€346.33,true
3,4,Re-contextualized interactive strategy,Electronics,394.82,1
4,5,Multi-tiered bottom-line implementation,Sports,433.18 EUR,no
5,6,Business-focused intermediate portal,Home,324.49,1
6,7,Exclusive bifurcated emulation,Toys,€314.29,1
7,8,Team-oriented mission-critical info-mediaries,Clothing,53.69,true
8,9,Enhanced systemic challenge,Sports,62.39,true
9,10,Face-to-face regional attitude,Toys,"323,37",1


=== products_clean: 100 rows ===


,product_id,name,category,price_eur,in_stock
0,1,Object-based explicit attitude,Clothing,134.94,True
1,2,Configurable upward-trending complexity,Clothing,498.06,True
2,3,Triple-buffered global hierarchy,Books,346.33,True
3,4,Re-contextualized interactive strategy,Electronics,394.82,True
4,5,Multi-tiered bottom-line implementation,Sports,433.18,False
5,6,Business-focused intermediate portal,Home,324.49,True
6,7,Exclusive bifurcated emulation,Toys,314.29,True
7,8,Team-oriented mission-critical info-mediaries,Clothing,53.69,True
8,9,Enhanced systemic challenge,Sports,62.39,True
9,10,Face-to-face regional attitude,Toys,323.37,True


=== products_raw: 0 duplicate rows (subset=all columns) ===


,product_id,name,category,price_eur,in_stock


In [14]:
# Orders

show_all(orders.sort_values("order_id"), "orders")          #sorted
#show_all(orders, "orders")                                 #unsorted

# Duplicates

inspect_duplicates(orders, name="orders")
#inspect_duplicates(orders, subset=["order_id"], name="orders")

NameError: name 'orders' is not defined